# Обновление населения
Население исправлено по локальному OSM и контекстным признакам жилья, включая частную застройку.
Актуальные результаты: 1 197 166 модельных жителей, 82 здания школ, 106 942 оценочных места, потребность 146 028, дефицит 54 209.
Это оценки, не перепись. Неустановленные здания и отсутствие зданий в OSM помечены отдельно.
Повторный пересчёт населения: `python examples/recalculate_population.py`. Школьный импорт теперь сохраняет исправленное население.


# Практика 2 — Тюмень, обновлённая школьная сеть

Новая версия `python-package-boilerplate-updated`. Старый проект сохранён отдельно.
Основной источник школ — `data_tyumen/refined/tyumen_schools_82.csv`: **82 здания, 106 942 оценочных места**.

Территория, 12 805 кварталов, модельное население, пешеходная матрица, морфотипы, землепользование и плотности переиспользуются без изменения. Повторно рассчитываются только школьная агрегация и обеспеченность.

**Ограничение:** значения вместимости включают проектные данные, численность учащихся, деление между корпусами, медиану и оценки. Не все значения являются подтверждённой проектной мощностью. Исходные методы, источники и достоверность сохранены в CSV и GeoJSON.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, IFrame

EXAMPLES = Path.cwd()
if (EXAMPLES / 'examples' / 'update_schools_csv.py').exists():
    EXAMPLES = EXAMPLES / 'examples'
sys.path.insert(0, str(EXAMPLES))
from update_schools_csv import DATA, PREVIOUS, load_csv, run_update

schools_csv = load_csv(DATA / 'tyumen_schools_82.csv')
print('Записей:', len(schools_csv), '| Оценочных мест:', int(schools_csv.capacity.sum()))
display(schools_csv[['name', 'building', 'address', 'capacity', 'capacity_method']].head(12))

## Обновление расчёта

Каждый корпус привязывается к одному кварталу исходным пространственным объединением BlocksNet. Координаты не смещаются; старые объекты OSM не добавляются. В модель передаётся суммарная **capacity** каждого квартала.

Метод: `competitive_provision`, BlocksNet 1.0.0a9, потребность `ceil(population / 1000 * 120)`, доступность 15 минут пешком, `self_supply=True`, `max_depth=1`. Нулевой контроль вместимости проверяет, что места участвуют в расчёте.


In [ ]:
schools, schools_blocks, provision, results = run_update()
validation = json.loads((DATA / 'schools_csv_validation.json').read_text(encoding='utf-8'))
display(pd.DataFrame(list(validation['checks'].items()), columns=['Проверка', 'Пройдена']))
display(schools[['name', 'building', 'capacity', 'capacity_method', 'block_id']].head(12))

In [ ]:
old = pd.read_csv(PREVIOUS / 'provision_schools.csv', index_col=0)
comparison = pd.DataFrame({
    'Версия': ['Прежняя школьная сеть', 'Новый CSV'],
    'Школьных точек': [70, len(schools)],
    'Оценочная вместимость, мест': [int(old.capacity.sum()), int(provision.capacity.sum())],
    'Потребность, мест': [int(old.demand.sum()), int(provision.demand.sum())],
    'Обслужено в пределах 15 минут, мест': [int(old.demand_within.sum()), int(provision.demand_within.sum())],
    'Модельный дефицит, мест': [int(old.demand_left.sum()), int(provision.demand_left.sum())],
})
display(comparison)
print('Разница обусловлена заменой исходных данных, это не эффект строительства новых школ.')
print('Предупреждения:')
for warning in validation['warnings']:
    print('—', warning)

In [ ]:
import runpy
runpy.run_path(str(EXAMPLES / 'tyumen_dashboard_refined.py'), run_name='__main__')
print('Готовый дашборд:', EXAMPLES / 'data_tyumen' / 'dashboard_refined.html')
display(IFrame(src='data_tyumen/dashboard_refined.html', width='100%', height=720))

## Результат

Новый набор полностью заменяет старые школьные точки. Все 82 здания отображаются на карте; вместимость поступает в пространственный расчёт. Нешкольные показатели сохранены из предыдущей работы без изменения. При повторном запуске CSV загружается заново как полный набор, поэтому точки не накапливаются.

Основные файлы проверки: `schools_import_audit.csv`, `schools_csv_validation.json`, `quality_report.json` в `data_tyumen/refined`.
